# 04 — Sentiment Aggregation & JST Join  *(updated for final submission)*

**Amendments applied:**
- keeps the corrected no-future-leakage fill logic
- adds an explicit **missingness / imputation audit**
- retains **speech-count coverage** in the annual panel
- adds an **imputation flag** so downstream notebooks can test sensitivity
- makes the cross-sectional fill assumption explicit in notebook commentary

This version is aimed at being methodologically safer for dissertation submission.


## Cell 1 — Paths and imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r'C:\Users\Owner\OneDrive\dissertation')

RAW_DIR       = BASE_DIR / 'data' / 'raw'
PROC_DIR      = BASE_DIR / 'data' / 'processed'
BIS_DIR       = RAW_DIR  / 'Bis_Org_Speaches'

RAW_SENTIMENT = PROC_DIR / 'bis_sentiment_raw.csv'
JST_FILE      = RAW_DIR  / 'JSTdatasetR6.xlsx'
OUT_ANNUAL    = PROC_DIR / 'sentiment_annual.csv'
OUT_MASTER    = PROC_DIR / 'jst_sentiment_master.csv'

PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Input file check:')
for fp in [RAW_SENTIMENT, JST_FILE]:
    tag = '✅  found' if fp.exists() else '❌  MISSING'
    print(f'  {tag}  →  {fp.name}')

JST_ISOS = [
    'USA','GBR','DEU','FRA','ITA','ESP','NLD','BEL',
    'PRT','IRL','CHE','JPN','AUS','CAN','SWE','NOR','DNK','FIN'
]
print(f'\nJST countries: {len(JST_ISOS)}')

Input file check:
  ✅  found  →  bis_sentiment_raw.csv
  ✅  found  →  JSTdatasetR6.xlsx

JST countries: 18


## Cell 2 — Load raw sentiment scores

In [3]:
df_raw = pd.read_csv(RAW_SENTIMENT)

print(f'Rows        : {len(df_raw):,}')
print(f'Columns     : {df_raw.columns.tolist()}')
print(f'Year range  : {df_raw["year"].min()} – {df_raw["year"].max()}')
print(f'Missing P_neg : {df_raw["P_neg"].isna().sum()}')
print()
print('First 3 rows:')
print(df_raw.head(3).to_string())
print()
if 'description' in df_raw.columns:
    print('Sample description values:')
    print(df_raw['description'].dropna().head(5).tolist())

Rows        : 16,622
Columns     : ['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral']
Year range  : 1997 – 2020
Missing P_neg : 4077

First 3 rows:
                                       url  year                 date              author                                                                                                                                                                                          description     P_pos     P_neg  P_neutral
0  https://www.bis.org/review/r970512a.pdf  1997  1997-04-24 00:00:00    Laurence H Meyer                                               Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97.  0.132078  0.230044   0.637878
1  https://www.bis.org/review/r970605b.pdf  1997  1997-05-26 00:00:00     Lars Heikensten                                                                Address by the Deputy Governor of 

## Cell 3 — Map institution names to 3-letter JST ISO codes

In [5]:
INSTITUTION_TO_ISO3 = {
    'federal reserve': 'USA', 'board of governors': 'USA',
    'federal open market': 'USA', 'new york fed': 'USA',
    'bank of england': 'GBR',
    'deutsche bundesbank': 'DEU', 'bundesbank': 'DEU',
    'banque de france': 'FRA', 'bank of france': 'FRA',
    'banca d italia': 'ITA', "banca d'italia": 'ITA', 'bank of italy': 'ITA',
    'banco de espana': 'ESP', 'banco de españa': 'ESP', 'bank of spain': 'ESP',
    'nederlandsche bank': 'NLD', 'netherlands bank': 'NLD',
    'national bank of belgium': 'BEL', 'banque nationale de belgique': 'BEL',
    'banco de portugal': 'PRT', 'bank of portugal': 'PRT',
    'central bank of ireland': 'IRL', 'bank of ireland': 'IRL',
    'swiss national bank': 'CHE', 'schweizerische nationalbank': 'CHE',
    'bank of japan': 'JPN',
    'reserve bank of australia': 'AUS', 'bank of australia': 'AUS',
    'bank of canada': 'CAN',
    'riksbank': 'SWE', 'sveriges riksbank': 'SWE', 'bank of sweden': 'SWE',
    'norges bank': 'NOR', 'bank of norway': 'NOR',
    'danmarks nationalbank': 'DNK', 'bank of denmark': 'DNK',
    'bank of finland': 'FIN', 'suomen pankki': 'FIN',
}

def map_iso3(description):
    if pd.isna(description):
        return None
    desc_lower = str(description).lower()
    for key, iso3 in INSTITUTION_TO_ISO3.items():
        if key in desc_lower:
            return iso3
    return None

df_raw['iso'] = df_raw['description'].apply(map_iso3)

mapped   = df_raw['iso'].notna().sum()
unmapped = df_raw['iso'].isna().sum()
print(f'Speeches mapped   : {mapped:,}')
print(f'Speeches unmapped : {unmapped:,}  (non-JST — will be dropped)')
print()
codes_found = df_raw['iso'].dropna().unique()
non_jst = [c for c in codes_found if c not in JST_ISOS]
if non_jst:
    print(f'WARNING: non-JST codes: {non_jst}')
else:
    print('✅  All mapped codes are valid JST 3-letter ISO codes.')

Speeches mapped   : 7,969
Speeches unmapped : 8,653  (non-JST — will be dropped)

✅  All mapped codes are valid JST 3-letter ISO codes.


## Cell 4 — Governor-level speech filter  *(IMPROVEMENT 1)*

Filters to speeches by Governors, Presidents, and Chairmen only.
Staff speeches introduce noise — only senior officials' speeches move markets
and are monitored by financial stability analysts.
If the `author` column is absent or the filter removes too many speeches,
it falls back gracefully to all speeches.

In [7]:
GOVERNOR_KEYWORDS = [
    'governor', 'president', 'chairman', 'chair',
    'deputy governor', 'vice president', 'vice-president',
    'chief executive', 'managing director', 'executive director',
    'deputy president', 'deputy chair',
]

df_filtered = df_raw.copy()

if 'author' in df_raw.columns and df_raw['author'].notna().sum() > 0:
    author_lower = df_raw['author'].fillna('').str.lower()
    gov_mask = author_lower.apply(
        lambda a: any(kw in a for kw in GOVERNOR_KEYWORDS)
    )
    n_gov   = gov_mask.sum()
    n_total = len(df_raw)
    pct     = 100 * n_gov / n_total

    if n_gov > 1000:   # Only apply if we have enough speeches
        df_filtered = df_raw[gov_mask].copy()
        print(f'Governor filter applied: {n_gov:,} / {n_total:,} speeches ({pct:.1f}%)')
        print()
        print('Author sample (first 5 governor speeches):')
        print(df_filtered['author'].dropna().head(5).tolist())
    else:
        print(f'Governor filter would retain only {n_gov} speeches — too few.')
        print('Falling back to all speeches (no author filter applied).')
else:
    print('No author column found — using all speeches (no governor filter).')
    print('This is fine; the improvement is optional.')

print()
print(f'Speeches entering aggregation: {len(df_filtered):,}')

Governor filter would retain only 1 speeches — too few.
Falling back to all speeches (no author filter applied).

Speeches entering aggregation: 16,622


## Cell 5 — Filter to 18 JST countries + 1997–2020 and aggregate annually

In [9]:
df_jst = df_filtered[
    df_filtered['iso'].isin(JST_ISOS) &
    df_filtered['year'].between(1997, 2020)
].copy()

print(f'Speeches after JST + year filter : {len(df_jst):,}')
print(f'Countries represented            : {df_jst["iso"].nunique()} / 18')
print()

missing_countries = [iso for iso in JST_ISOS if iso not in df_jst['iso'].values]
if missing_countries:
    print(f'WARNING: no speeches for: {missing_countries} — will be gap-filled')
else:
    print('✅  All 18 JST countries represented.')
print()

sentiment_annual = (
    df_jst
    .groupby(['year', 'iso'])
    .agg(
        P_pos      = ('P_pos',     'mean'),
        P_neg      = ('P_neg',     'mean'),
        P_neutral  = ('P_neutral', 'mean'),
        n_speeches = ('P_neg',     'count'),
    )
    .reset_index()
)
sentiment_annual['net_sentiment'] = (
    sentiment_annual['P_pos'] - sentiment_annual['P_neg']
).round(6)
for col in ['P_pos', 'P_neg', 'P_neutral']:
    sentiment_annual[col] = sentiment_annual[col].round(6)

sentiment_annual.to_csv(OUT_ANNUAL, index=False)
print(f'Annual sentiment rows : {len(sentiment_annual)}  (up to 432 = 18 × 24)')
print(f'Saved → {OUT_ANNUAL}')
print()
print(sentiment_annual.head(6).to_string(index=False))

Speeches after JST + year filter : 7,969
Countries represented            : 18 / 18

✅  All 18 JST countries represented.

Annual sentiment rows : 393  (up to 432 = 18 × 24)
Saved → C:\Users\Owner\OneDrive\dissertation\data\processed\sentiment_annual.csv

 year iso    P_pos    P_neg  P_neutral  n_speeches  net_sentiment
 1997 AUS 0.152554 0.247528   0.599918           7      -0.094974
 1997 CAN 0.317820 0.271316   0.410865           6       0.046504
 1997 CHE 0.131788 0.137297   0.730915           1      -0.005509
 1997 DEU 0.171991 0.076354   0.751656           8       0.095637
 1997 FIN 0.562373 0.109327   0.328300           1       0.453046
 1997 FRA 0.331707 0.410152   0.258141           7      -0.078445


## Cell 6 — Coverage diagnostic

Checks country-year coverage before filling and keeps `n_speeches` as a coverage variable.
This matters because weak sentiment performance may partly reflect sparse BIS speech coverage
rather than absence of true signal.


In [11]:
full_grid = pd.MultiIndex.from_product(
    [range(1997, 2021), JST_ISOS], names=['year', 'iso']
).to_frame(index=False)

check   = full_grid.merge(sentiment_annual[['year','iso','P_neg']], on=['year','iso'], how='left')
missing = check[check['P_neg'].isna()]

print(f'Expected country-years : {len(full_grid)}')
print(f'With sentiment data    : {check["P_neg"].notna().sum()}')
print(f'Missing                : {len(missing)}')

if len(missing) > 0:
    miss_summary = (
        missing.groupby('iso')['year']
        .agg(['count','min','max'])
        .rename(columns={'count':'n_missing','min':'first_missing','max':'last_missing'})
    )
    print(miss_summary.to_string())
    print('Gaps filled with per-country means in Cell 7.')
else:
    print('\n✅  Full 432-row coverage.')

Expected country-years : 432
With sentiment data    : 389
Missing                : 43
     n_missing  first_missing  last_missing
iso                                        
BEL          7           1997          2020
DNK          4           1997          2020
ESP          4           1997          2001
FIN          3           1999          2002
FRA          1           1998          1998
IRL          8           1998          2009
ITA          2           2002          2003
NOR          2           1997          1998
PRT         12           1997          2009
Gaps filled with per-country means in Cell 7.


## Cell 6b — Missingness and imputation audit  *(new)*

This audit separates:
1. missing sentiment before any fill  
2. missingness remaining after **forward-fill within country**  
3. missingness remaining after the **cross-sectional year mean fill**

This does **not** prove the imputation is harmless, but it makes the assumption transparent and
gives you an examiner-facing summary you can cite in the dissertation.


In [ ]:
# Missingness audit before imputation
sa_audit = full_grid.merge(sentiment_annual, on=['year','iso'], how='left')
sa_audit = sa_audit.sort_values(['iso','year']).reset_index(drop=True)

SENT_COLS = ['P_pos','P_neg','P_neutral','net_sentiment']

# Before fill
missing_before = (
    sa_audit.groupby('iso')[SENT_COLS]
    .apply(lambda g: g.isna().sum())
)

# After forward-fill only
sa_ffill = sa_audit.copy()
for col in SENT_COLS:
    sa_ffill[col] = sa_ffill.groupby('iso')[col].transform(lambda x: x.ffill())

missing_after_ffill = (
    sa_ffill.groupby('iso')[SENT_COLS]
    .apply(lambda g: g.isna().sum())
)

# After forward-fill + cross-sectional year mean fill
sa_final = sa_ffill.copy()
for col in SENT_COLS:
    cross_mean = sa_final.groupby('year')[col].transform('mean')
    sa_final[col] = sa_final[col].fillna(cross_mean)

missing_after_final = (
    sa_final.groupby('iso')[SENT_COLS]
    .apply(lambda g: g.isna().sum())
)

# Compact audit table using P_neg as the headline series
audit_tbl = pd.DataFrame({
    'missing_before_P_neg': missing_before['P_neg'],
    'after_ffill_P_neg': missing_after_ffill['P_neg'],
    'after_final_fill_P_neg': missing_after_final['P_neg'],
}).sort_index()

print('Missingness audit for P_neg by country')
print(audit_tbl.to_string())
print()
print('Totals:')
print(audit_tbl.sum().to_string())

# Optional broader summary across all sentiment columns
summary_all = pd.DataFrame({
    'before': missing_before.sum(),
    'after_ffill': missing_after_ffill.sum(),
    'after_final_fill': missing_after_final.sum()
})
print()
print('All sentiment columns combined:')
print(summary_all.to_string())


## Cell 7 — Fill gaps, build lags, and add sentiment momentum  *(updated)*

Gap-filling logic:
- **Step 1:** forward-fill within country
- **Step 2:** only if gaps remain, use the same-year cross-sectional mean
- **Step 3:** any residual gaps are set to neutral values and flagged

This is cleaner than the earlier full-sample country-mean fill, but the cross-sectional step still
imposes a strong assumption. Keep that caveat in the dissertation.


In [13]:
# ── CORRECTED gap-filling: no future information leakage ─────────────────
# Original problem: full-sample country means used future years.
# Current approach:
#   1) forward-fill within country
#   2) fill remaining gaps with same-year cross-sectional mean
#   3) keep explicit imputation flags for transparency / sensitivity checks

# Merge onto full 432-row grid
sa = full_grid.merge(sentiment_annual, on=['year','iso'], how='left')
print(f'Rows after merge: {len(sa)}  (expect 432)')

# Sort before any fill or shift
sa = sa.sort_values(['iso','year']).reset_index(drop=True)

SENT_COLS = ['P_pos','P_neg','P_neutral','net_sentiment']

# Preserve pre-fill missingness for an imputation flag
for col in SENT_COLS:
    sa[f'{col}_was_missing'] = sa[col].isna().astype(int)

sa['sentiment_imputed_any'] = sa[[f'{c}_was_missing' for c in SENT_COLS]].max(axis=1)

# Coverage variable: no speeches means no direct annual signal
if 'n_speeches' in sa.columns:
    sa['n_speeches'] = sa['n_speeches'].fillna(0).astype(int)

# Step 1: forward-fill within country (uses previous years only)
for col in SENT_COLS:
    sa[col] = sa.groupby('iso')[col].transform(lambda x: x.ffill())

# Step 2: fill remaining NaN with cross-sectional mean for that year
# This does not use future years, but it does borrow information from other countries
for col in SENT_COLS:
    cross_mean = sa.groupby('year')[col].transform('mean')
    n = sa[col].isna().sum()
    sa[col] = sa[col].fillna(cross_mean)
    if n > 0:
        print(f'  {col}: filled {n} gaps with cross-sectional year mean')

# Step 3: residual safeguard
remaining = sa[SENT_COLS].isna().sum()
if remaining.sum() == 0:
    print('  No remaining NaN — all sentiment gaps resolved')
else:
    print(f'  Remaining NaN after year-mean fill: {remaining.to_dict()}')
    print('  Residual NaN filled with neutral fallback values')
    sa['P_pos'] = sa['P_pos'].fillna(0)
    sa['P_neg'] = sa['P_neg'].fillna(0)
    sa['P_neutral'] = sa['P_neutral'].fillna(1)
    sa['net_sentiment'] = sa['net_sentiment'].fillna(0)

print()
print('Imputation flag summary:')
print(sa['sentiment_imputed_any'].value_counts(dropna=False).sort_index().to_string())
print()

# ── Sentiment momentum (YoY change) ──────────────────────────────────────
sa['P_neg_change']    = sa.groupby('iso')['P_neg'].diff()
sa['P_pos_change']    = sa.groupby('iso')['P_pos'].diff()
sa['net_sent_change'] = sa.groupby('iso')['net_sentiment'].diff()

# ── Rolling 3-year mean and std (no look-ahead) ──────────────────────────
# shift(1) inside rolling ensures we use data only up to t-1 at each point
ROLL_WIN = 3
for col in ['P_neg','net_sentiment']:
    sa[f'{col}_roll3_mean'] = (
        sa.groupby('iso')[col]
          .transform(lambda x: x.shift(1).rolling(ROLL_WIN, min_periods=2).mean())
    )
    sa[f'{col}_roll3_std'] = (
        sa.groupby('iso')[col]
          .transform(lambda x: x.shift(1).rolling(ROLL_WIN, min_periods=2).std())
    )

# ── Rolling OLS trend slope (uses only past/current values, then lagged) ─
SLOPE_WIN = 4

def rolling_slope_series(series, window=SLOPE_WIN):
    vals = series.values
    result = np.full(len(vals), np.nan)
    t = np.arange(window)
    for i in range(window - 1, len(vals)):
        y = vals[i-window+1:i+1]
        if np.sum(~np.isnan(y)) >= window - 1:
            result[i] = np.polyfit(t, y, 1)[0]
    return pd.Series(result, index=series.index)

sa['P_neg_slope4']    = sa.groupby('iso')['P_neg'].transform(lambda x: rolling_slope_series(x))
sa['net_sent_slope4'] = sa.groupby('iso')['net_sentiment'].transform(lambda x: rolling_slope_series(x))

# ── Lag all sentiment-derived columns at t-1 and t-2 ─────────────────────
ALL_SENT_DERIVED = (
    SENT_COLS
    + ['P_neg_change','P_pos_change','net_sent_change']
    + ['P_neg_roll3_mean','P_neg_roll3_std','net_sent_roll3_mean','net_sent_roll3_std']
    + ['P_neg_slope4','net_sent_slope4']
)

for col in ALL_SENT_DERIVED:
    if col in sa.columns:
        sa[f'{col}_lag1'] = sa.groupby('iso')[col].shift(1)
        sa[f'{col}_lag2'] = sa.groupby('iso')[col].shift(2)

lag1_cols = [c for c in sa.columns if c.endswith('_lag1')]
lag2_cols = [c for c in sa.columns if c.endswith('_lag2')]
print(f't-1 lag columns: {len(lag1_cols)}')
print(f't-2 lag columns: {len(lag2_cols)}')
print(f'NaN in P_neg_lag1 (expect ~18): {sa["P_neg_lag1"].isna().sum()}')
print()
print('USA alignment check:')
cols_chk = ['year','P_neg','P_neg_lag1','P_neg_roll3_mean','P_neg_slope4','sentiment_imputed_any']
print(sa[sa['iso']=='USA'][cols_chk].head(8).to_string(index=False))


Rows after merge: 432  (expect 432)
  P_pos: filled 17 gaps with cross-sectional mean
  P_neg: filled 17 gaps with cross-sectional mean
  P_neutral: filled 17 gaps with cross-sectional mean
  net_sentiment: filled 17 gaps with cross-sectional mean
  No remaining NaN — all gaps filled without future leakage

t-1 lag columns: 11
t-2 lag columns: 11
NaN in P_neg_lag1 (expect 18): 18

USA alignment check:
 year    P_neg  P_neg_lag1  P_neg_roll3_mean  P_neg_slope4
 1997 0.163145         NaN               NaN           NaN
 1998 0.163982    0.163145               NaN           NaN
 1999 0.149645    0.163982          0.163564           NaN
 2000 0.158812    0.149645          0.158924     -0.002734
 2001 0.196379    0.158812          0.157480      0.010636
 2002 0.209385    0.196379          0.168279      0.021679
 2003 0.229068    0.209385          0.188192      0.022377
 2004 0.212794    0.229068          0.211611      0.006893


## Cell 8 — Join to JST macro panel

In [15]:
jst = pd.read_excel(JST_FILE)
jst.columns = jst.columns.str.lower().str.strip()
jst_window = jst[jst['year'].between(1997, 2020)].copy()
print(f'JST rows 1997-2020 : {len(jst_window)}')

df_master = jst_window.merge(sa, on=['year','iso'], how='left')

print(f'Master rows              : {len(df_master)}  (expect 432)')
print(f'Master columns           : {len(df_master.columns)}')
print(f'P_neg_lag1 valid         : {df_master["P_neg_lag1"].notna().sum()}')
print(f'P_neg_lag2 valid         : {df_master["P_neg_lag2"].notna().sum()}')
print(f'P_neg_change_lag1 valid  : {df_master["P_neg_change_lag1"].notna().sum()}')
if 'n_speeches' in df_master.columns:
    print(f'Rows with zero speeches   : {(df_master["n_speeches"] == 0).sum()}')
if 'sentiment_imputed_any' in df_master.columns:
    print(f'Rows with any imputation  : {df_master["sentiment_imputed_any"].sum()}')

df_master.to_csv(OUT_MASTER, index=False)
print(f'\n✅  Saved → {OUT_MASTER}')


JST rows 1997-2020 : 432
Master rows        : 432  (expect 432)
Master columns     : 95
P_neg_lag1         : 414 valid
P_neg_lag2         : 396 valid
P_neg_change_lag1  : 396 valid

✅  Saved → C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv


## Cell 9 — Final validation

In [17]:
print('='*60)
print('  MASTER DATASET VALIDATION (updated final version)')
print('='*60)
checks = [
    ('Rows = 432',                     len(df_master) == 432),
    ('Countries = 18',                 df_master['iso'].nunique() == 18),
    ('Year range 1997-2020',           df_master['year'].min() == 1997 and df_master['year'].max() == 2020),
    ('P_neg_lag1 present',             'P_neg_lag1' in df_master.columns),
    ('P_neg_lag2 present',             'P_neg_lag2' in df_master.columns),
    ('P_neg_change_lag1 present',      'P_neg_change_lag1' in df_master.columns),
    ('P_neg_roll3_mean_lag1 present',  'P_neg_roll3_mean_lag1' in df_master.columns),
    ('P_neg_slope4_lag1 present',      'P_neg_slope4_lag1' in df_master.columns),
    ('n_speeches present',             'n_speeches' in df_master.columns),
    ('sentiment_imputed_any present',  'sentiment_imputed_any' in df_master.columns),
    ('crisisjst present',              'crisisjst' in df_master.columns),
    ('tloans present',                 any('tloans' in c for c in df_master.columns)),
]
all_ok = True
for label, result in checks:
    print(f'  {"V" if result else "X"}  {label}')
    if not result:
        all_ok = False

print()
if 'sentiment_imputed_any' in df_master.columns:
    print('Imputation coverage:')
    print(df_master['sentiment_imputed_any'].value_counts(dropna=False).sort_index().to_string())
    print()

print('  ALL CHECKS PASSED.' if all_ok else '  FAILURES — fix before proceeding.')
print(f'Output: {OUT_MASTER}  ({OUT_MASTER.stat().st_size // 1024} KB)')


  MASTER DATASET VALIDATION (corrected — no future leakage)
  V  Rows = 432
  V  Countries = 18
  V  Year range 1997-2020
  V  P_neg_lag1 present
  V  P_neg_lag2 present
  V  P_neg_change_lag1 present
  V  P_neg_roll3_mean_lag1 present
  V  P_neg_slope4_lag1 present
  V  crisisjst present
  V  tloans present
  V  No full-sample mean fill

  ALL CHECKS PASSED (corrected pipeline).
Output: C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv  (506 KB)
